In [11]:
!apt -y install ffmpeg
!pip -q install faster-whisper==1.0.3 soundfile==0.12.1 numpy pandas matplotlib scikit-learn


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.


In [18]:
from google.colab import files
uploaded = files.upload()  # select the 3–5 .wav files from marathi_samples


Saving mrt_09697_01834070454.wav to mrt_09697_01834070454.wav
Saving mrt_09697_00476676389.wav to mrt_09697_00476676389.wav
Saving mrt_04310_00857386910.wav to mrt_04310_00857386910.wav
Saving mrt_04310_00952396785.wav to mrt_04310_00952396785.wav
Saving mrt_04310_01296588961.wav to mrt_04310_01296588961.wav
Saving mrt_04310_00097208089.wav to mrt_04310_00097208089.wav
Saving mrt_04310_00331730744.wav to mrt_04310_00331730744.wav
Saving mrt_03398_01675504879.wav to mrt_03398_01675504879.wav
Saving mrt_03397_01906514832.wav to mrt_03397_01906514832.wav
Saving mrt_03397_00942481581.wav to mrt_03397_00942481581.wav
Saving mrt_03397_01056594829.wav to mrt_03397_01056594829.wav
Saving mrt_03397_01143771909.wav to mrt_03397_01143771909.wav
Saving mrt_03397_00129434087.wav to mrt_03397_00129434087.wav
Saving mrt_03349_00644171543.wav to mrt_03349_00644171543.wav
Saving mrt_03349_01688384300.wav to mrt_03349_01688384300.wav
Saving mrt_02624_01527619254.wav to mrt_02624_01527619254.wav
Saving m

In [25]:
# Cell 1 — Load model (CPU-friendly)
from faster_whisper import WhisperModel

try:
    model
    print("Model already loaded.")
except NameError:
    model = WhisperModel("small", device="cpu", compute_type="int8")
    print("Model loaded.")


# Cell 1B — Optional: load medium model for higher accuracy
from faster_whisper import WhisperModel

try:
    medium_model
    print("Medium model already loaded.")
except NameError:
    print("Loading medium model (may take ~2–3 min on Colab CPU)...")
    medium_model = WhisperModel("medium", device="cpu", compute_type="int8")
    print("Medium model loaded.")


Model already loaded.
Loading medium model (may take ~2–3 min on Colab CPU)...


config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

vocabulary.txt: 0.00B [00:00, ?B/s]

model.bin:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

Medium model loaded.


In [27]:
# Cell 2 — Helpers (updated)
import io, soundfile as sf
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Audio, display

# Cell 2B — use medium model
def predict_language_distribution_from_path_medium(path: str):
    segments, info = medium_model.transcribe(
        path,
        vad_filter=True,
        beam_size=1,
        task="transcribe",
        language=None
    )
    probs = getattr(info, "language_probs", None)
    return info.language, float(info.language_probability), probs


# (Optional) bytes version — only if you need it elsewhere
def predict_language_distribution_from_bytes(raw_bytes: bytes):
    # decode bytes -> float32 mono array using soundfile, then pass array
    buf = io.BytesIO(raw_bytes)
    audio, sr = sf.read(buf, dtype="float32", always_2d=False)
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    # faster-whisper can take a float32 array directly
    segments, info = model.transcribe(
        audio,
        vad_filter=True,
        beam_size=1,
        task="transcribe",
        language=None
    )
    probs = getattr(info, "language_probs", None)
    return info.language, float(info.language_probability), probs

def plot_probs(distro: dict, title="Top-k language probabilities", topn=8):
    if not isinstance(distro, dict) or len(distro) == 0:
        print("Full probability distribution not available (showing top-1 in text only).")
        return
    items = sorted(distro.items(), key=lambda x: x[1], reverse=True)[:topn]
    labels = [k for k, _ in items]
    vals = [float(v) for _, v in items]
    plt.figure(figsize=(5,3))
    plt.bar(labels, vals)
    plt.ylim(0, 1.0)
    plt.ylabel("Probability")
    plt.title(title)
    plt.xticks(rotation=30)
    plt.show()


In [21]:
# Cell 3 — Get uploaded files (reuse if present)
from google.colab import files

try:
    uploaded  # reuse previous upload dict
    if not isinstance(uploaded, dict) or len(uploaded) == 0:
        raise NameError
    print(f"Reusing {len(uploaded)} uploaded file(s).")
except NameError:
    print("Please select your .wav/.mp3/.m4a files again:")
    uploaded = files.upload()
    print(f"Uploaded {len(uploaded)} file(s).")


Reusing 25 uploaded file(s).


In [28]:
# Cell 4 — Only predict on 5 random files in /content/
import os, random
from IPython.display import Audio, display

# Change these values anytime
BASE_DIR = "/content"
NUM_DEMO = 5
SHUFFLE  = True

# collect audio files from /content
all_files = [os.path.join(BASE_DIR, f) for f in os.listdir(BASE_DIR)
             if f.lower().endswith((".wav", ".mp3", ".m4a"))]

if not all_files:
    raise RuntimeError("No audio files found in /content/ — upload first.")

if SHUFFLE:
    random.shuffle(all_files)

demo_files = all_files[:NUM_DEMO]

results = []

print(f"Running demo on {len(demo_files)} file(s):")
for path in demo_files:
    fname = os.path.basename(path)
    print("—"*80)
    print(f"File: {path}")

    try:
        display(Audio(path))
    except Exception as e:
        print("Could not preview audio:", e)

    # ✅ Use the medium model (if you added Cell 1B + 2B)
    lang, conf, distro = predict_language_distribution_from_path_medium(path)

    print(f"Predicted language: {lang}  (confidence ≈ {conf:.2f})")
    results.append((fname, lang, conf))
    plot_probs(distro, title=f"{fname} — top-k probs")

print("—"*80)
print("Done.")


Running demo on 5 file(s):
————————————————————————————————————————————————————————————————————————————————
File: /content/mrt_02624_00090780723.wav


Predicted language: mr  (confidence ≈ 0.76)
Full probability distribution not available (showing top-1 in text only).
————————————————————————————————————————————————————————————————————————————————
File: /content/mrt_02436_01552526138.wav


Predicted language: hi  (confidence ≈ 0.31)
Full probability distribution not available (showing top-1 in text only).
————————————————————————————————————————————————————————————————————————————————
File: /content/mrt_02436_00154358097.wav


Predicted language: hi  (confidence ≈ 0.45)
Full probability distribution not available (showing top-1 in text only).
————————————————————————————————————————————————————————————————————————————————
File: /content/mrt_04310_01296588961.wav


Predicted language: hi  (confidence ≈ 0.60)
Full probability distribution not available (showing top-1 in text only).
————————————————————————————————————————————————————————————————————————————————
File: /content/mrt_03397_01906514832.wav


Predicted language: mr  (confidence ≈ 0.93)
Full probability distribution not available (showing top-1 in text only).
————————————————————————————————————————————————————————————————————————————————
Done.


Summary Table


In [29]:
# Cell 5 — Summary table
import pandas as pd

# results = [(filename, lang_code, confidence), ...] from Cell 4
df = pd.DataFrame(results, columns=["filename", "predicted_code", "confidence"])

code2name = {
    "mr":"Marathi","bn":"Bengali","si":"Sinhala","hi":"Hindi","gu":"Gujarati","ta":"Tamil","te":"Telugu",
    "pa":"Punjabi","ur":"Urdu","ne":"Nepali","or":"Odia","as":"Assamese","kn":"Kannada","ml":"Malayalam",
    "sd":"Sindhi","ks":"Kashmiri","sa":"Sanskrit","en":"English","es":"Spanish","fr":"French","de":"German"
}

df["predicted_lang"] = df["predicted_code"].map(code2name).fillna(df["predicted_code"])
display(df)

print("\nCounts by predicted language:")
counts = df["predicted_lang"].value_counts().rename_axis("language").reset_index(name="count")
display(counts)


,filename,predicted_code,confidence,predicted_lang
0,mrt_02624_00090780723.wav,mr,0.763880,Marathi
1,mrt_02436_01552526138.wav,hi,0.314501,Hindi
2,mrt_02436_00154358097.wav,hi,0.445112,Hindi
3,mrt_04310_01296588961.wav,hi,0.603001,Hindi
4,mrt_03397_01906514832.wav,mr,0.929061,Marathi



Counts by predicted language:


,language,count
0,Hindi,3
1,Marathi,2


In [30]:
# Cell 6 to highlight confusable Indo-Aryan pairs

confusable = {
    "Marathi": {"Hindi","Bengali","Sinhala","Gujarati"},
    "Hindi":   {"Marathi","Bengali","Punjabi"},
    "Bengali": {"Assamese","Hindi","Marathi"},
    "Sinhala": {"Marathi","Tamil"}
}

def tag_confusables(lang_name):
    for base, near in confusable.items():
        if lang_name == base or lang_name in near:
            return base
    return None

df["confusable_family"] = df["predicted_lang"].apply(tag_confusables)
display(df[["filename","predicted_lang","confidence","confusable_family"]])


,filename,predicted_lang,confidence,confusable_family
0,mrt_02624_00090780723.wav,Marathi,0.763880,Marathi
1,mrt_02436_01552526138.wav,Hindi,0.314501,Marathi
2,mrt_02436_00154358097.wav,Hindi,0.445112,Marathi
3,mrt_04310_01296588961.wav,Hindi,0.603001,Marathi
4,mrt_03397_01906514832.wav,Marathi,0.929061,Marathi


In [31]:
# Cell 10 — Optional: measure per-file latency with your current model
import time

latencies = []
for fname, _, _ in results:
    start = time.time()
    # If you used the medium model:
    lang, conf, _ = predict_language_distribution_from_path_medium(f"/content/{fname}" if not fname.startswith("/content/") else fname)
    # If using small model instead, swap to:
    # lang, conf, _ = predict_language_distribution_from_path(f"/content/{fname}" if not fname.startswith("/content/") else fname)
    latencies.append(time.time() - start)

if latencies:
    import numpy as np
    print(f"Avg latency: {np.mean(latencies):.2f}s | Min: {np.min(latencies):.2f}s | Max: {np.max(latencies):.2f}s over {len(latencies)} file(s)")


Avg latency: 14.06s | Min: 13.71s | Max: 14.55s over 5 file(s)
